# Limpieza de datos.
## Grupo 2

### Importación de librerías.

In [29]:
import pandas as pd
import numpy as np

### Importación de datos.

In [30]:
df = pd.read_csv('train.csv', index_col=0)
df_labels = pd.read_csv('train_labels.csv', index_col=0)
labels = pd.read_csv('target_pairs.csv')

### Aplicando escala log a df.

In [31]:
df_log = np.log(df)
df_log.head(5)

,LME_AH_Close,LME_CA_Close,LME_PB_Close,LME_ZS_Close,JPX_Gold_Mini_Futures_Open,JPX_Gold_Rolling-Spot_Futures_Open,JPX_Gold_Standard_Futures_Open,JPX_Platinum_Mini_Futures_Open,JPX_Platinum_Standard_Futures_Open,JPX_RSS3_Rubber_Futures_Open,...,FX_GBPCAD,FX_CADCHF,FX_NZDCAD,FX_NZDCHF,FX_ZAREUR,FX_NOKGBP,FX_NOKCHF,FX_ZARCHF,FX_NOKJPY,FX_ZARGBP
date_id,,,,,,,,,,,,,,,,,,,,,
0,7.725109,8.882531,7.851661,8.116417,NaN,NaN,NaN,NaN,NaN,NaN,...,0.530621,-0.252477,-0.118654,-0.371130,-2.708255,-2.401500,-2.123352,-2.549317,2.626315,-2.827459
1,7.708860,8.874448,7.855157,8.109826,NaN,NaN,NaN,NaN,NaN,NaN,...,0.527847,-0.250153,-0.117109,-0.367262,-2.697793,-2.393637,-2.115940,-2.537472,2.631036,-2.815162
2,7.718685,8.880238,7.858254,8.120291,8.451908,8.453401,8.451908,8.120589,8.121777,5.332719,...,0.526339,-0.248223,-0.112045,-0.360267,-2.697199,-2.391657,-2.113544,-2.534681,2.637891,-2.812794
3,7.697348,8.870803,7.839919,8.117909,8.461258,8.463159,8.461469,8.140316,8.139149,5.332719,...,0.520644,-0.241653,-0.117164,-0.358817,-2.693571,-2.390783,-2.111791,-2.534706,2.641595,-2.813694
4,7.684784,8.871365,7.864804,8.127405,NaN,NaN,NaN,NaN,NaN,NaN,...,0.521656,-0.239192,-0.115364,-0.354556,-2.696472,-2.393977,-2.111510,-2.539257,2.640040,-2.821729


### Seprando las etiquetas del conjunto lables.

In [32]:
labels['etiquetas'] = [x.split(' - ') for x in labels['pair']]

### Función que permite reemplazar valores nulos de df_labels mediante cálculo de df_log.

In [33]:
def nulos_calculables():

    # Identifica cuántos valores fueron alterados.
    contador = 0

    # Se recorren todas las columnas a operar
    for columna in df_labels.columns:

        # Explicación de cómo calcular los datos.
        variables = labels.loc[labels['target'] == columna, 'etiquetas'].iloc[0]
        lag = labels.loc[labels['target'] == columna, 'lag'].iloc[0]

        # Se evalúa si es requerido y posible un cambio.
        for i in range(1, len(columna)-lag-1):
            if pd.isna(df_labels.loc[i, columna]):
                if df_log.loc[i-1:i, variables].isna().all().all():

                    # Se realiza el cambio.
                    parte_1 = df_log.loc[i+lag-1, variables[0]]- df_log.loc[i-1, variables[0]]
                    if len(variables) == 2:
                        parte_2 = df_log.loc[i+lag-1, variables[1]]- df_log.loc[i-1, variables[1]]
                        df_labels.loc[i, columna] = parte_1 - parte_2
                    else:
                        df_labels.loc[i, columna] = parte_1

                    # Aumenta el contador.
                    contador += 1

    print('Se realizaron', contador, 'cambios a valores nulos.')
nulos_calculables()

Se realizaron 0 cambios a valores nulos.


### Función que permite procesar los valores nulos al inicio y al final de df_logs.

In [34]:
def nulos_inicio_fin():

    cambios = 0
    n, m = df_log.shape

    # Procesando si el primer valor es nulo
    for columna in df_log.columns:
        contador = 0

        if pd.isna(df_log.loc[contador, columna]):
            while pd.isna(df_log.loc[contador, columna]):
                contador += 1
            valor_util = df_log.loc[contador, columna]
            for j in range(0, contador):
                df_log.loc[j, columna] = valor_util
                cambios += 1

    # Procesando si el último valor es nulo
    for columna in df_log.columns:
        contador = 1

        if pd.isna(df_log.loc[n-contador, columna]):
            while pd.isna(df_log.loc[n-contador, columna]):
                contador += 1
            valor_util = df_log.loc[n-contador, columna]
            for j in range(1, contador):
                df_log.loc[n-j, columna] = valor_util
                cambios += 1

    print('Se realizaron', cambios, 'cambios a valores nulos.')
nulos_inicio_fin()

Se realizaron 8750 cambios a valores nulos.


### Función que permite procesar mediante interpolación, los valores faltantes del conjunto df_log.